# 🎓 TeachRL v2 — Training Notebook

**Theme 4: Self-Improvement** | Meta PyTorch Hackathon x Scaler 2025

This notebook trains a PPO agent on the TeachRL v2 environment using **HuggingFace TRL**.

The agent must:
1. **Identify** which of 8 hidden student archetypes it is teaching
2. **Adapt** its curriculum strategy in real-time
3. **Improve** as the self-play loop generates harder student variants

**Links:**
- HF Space: https://huggingface.co/spaces/ArchedEquation/TeachRL
- GitHub: https://github.com/ArchedEquation/TeachRL

---

## 1. Install Dependencies

In [1]:
!pip install -q stable-baselines3 gymnasium pydantic fastapi uvicorn pyyaml openai wandb matplotlib

## 2. Clone Repository

In [2]:
!git clone https://github.com/ArchedEquation/TeachRL/content/TeachRL
%cd /content/TeachRL
!ls

Cloning into 'TeachRL'...
remote: Not Found
fatal: repository 'https://github.com/ArchedEquation/TeachRL/content/TeachRL/' not found
[Errno 2] No such file or directory: '/content/TeachRL'
/home/praneeth-yeddu/teachrl/teachrl2
app.py
baseline
Dockerfile
env
{env,graders,baseline,self_play,tests,server,training_plots}
graders
inference.py
models
models.py
openenv.yaml
pyproject.toml
README.md
requirements.txt
self_play
server
TeachRL_v2_Training.ipynb
tests
training_curves
training_plots
train_trl.py
uv.lock


## 3. Verify Environment Works

In [3]:
import sys
sys.path.insert(0, '/content/TeachRL')

from env.environment import TeachRLEnv, TASK_REGISTRY
from env.archetypes import ALL_ARCHETYPES

print(f'Tasks: {list(TASK_REGISTRY.keys())}')
print(f'Archetypes: {len(ALL_ARCHETYPES)}')

# Quick sanity check
env = TeachRLEnv(task_id='blind_teaching', seed=42, eval_mode=True)
obs = env.reset()
result = env.step({'concept': 'algebra_basics', 'difficulty': 'medium',
                   'hint_given': False, 'archetype_guess': None})
print(f'Step OK — reward={result.reward:.3f} done={result.done}')
print(f'True archetype: {env._sim.archetype_id.value}')
print(f'Expert hint: {obs.expert_hint[:60]}...')

Tasks: ['archetype_identification', 'adaptive_curriculum', 'blind_teaching', 'self_play_escalation']
Archetypes: 8
Step OK — reward=0.184 done=False
True archetype: overconfident_learner
Expert hint: Student appears confident but makes careless mistakes on kno...


## 4. Run Baseline Evaluation (Before Training)

In [ ]:
!python baseline/baseline_inference.py --episodes 10 --seed 42

## 5. (Optional) Login to W&B for Logging

In [ ]:
# Optional — skip if you don't want W&B logging
import wandb
wandb.login()

## 6. Train PPO — Easy Task (Archetype Identification)

In [4]:
!python train_trl.py \
    --task archetype_identification \
    --steps 100000 \
    --seed 42 \
    --eval


  TeachRL v2 — Training with HuggingFace TRL + PPO
  Tasks:  ['archetype_identification']
  Seed:   42
  W&B:    False

  Task:        archetype_identification
  Description: Easy — Identify hidden student archetype
  Timesteps:   100,000
  Parallel envs: 4
  n_steps/update: 512
/home/praneeth-yeddu/anaconda3/envs/adapt-tutor/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
  [archetype_identification] step= 10,000 | reward= 7.048 | fps= 2105 | elapsed=    5s
  [archetype_identification] step= 20,000 | reward= 

## 7. Train PPO — Medium Task (Adaptive Curriculum)

In [ ]:
!python train_trl.py \
    --task adaptive_curriculum \
    --steps 200000 \
    --seed 42 \
    --eval

## 8. Train PPO — Hard Task (Blind Teaching)

In [ ]:
!python train_trl.py \
    --task blind_teaching \
    --steps 400000 \
    --seed 42 \
    --eval

## 9. Train PPO — Expert Task (Self-Play Escalation)

This is the **Theme 4 core task**. The environment gets harder as the agent improves.

In [ ]:
!python train_trl.py \
    --task self_play_escalation \
    --steps 500000 \
    --seed 42 \
    --eval

## 10. Full Evaluation — PPO vs All Baselines

In [ ]:
!python baseline/rl_agent.py --eval --task all --episodes 10 --seed 42

## 11. Display Training Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

plots = sorted(glob.glob('training_plots/*.png'))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, plot_path in zip(axes.flat, plots[:4]):
    img = mpimg.imread(plot_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(plot_path.split('/')[-1].replace('.png','').replace('_',' ').title(),
                 fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('training_plots/all_plots_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Combined plot saved!')

## 12. Demo — Watch the Trained Agent Teach

See how the PPO agent behaves vs a random agent on the hard task.

In [ ]:
from stable_baselines3 import PPO
from env.environment import TeachRLEnv
from env.gym_wrapper import obs_to_vector, int_to_action
from env.environment import TASK_REGISTRY
import numpy as np

task_id   = 'blind_teaching'
max_steps = TASK_REGISTRY[task_id]['max_steps']

try:
    model = PPO.load(f'models/ppo_{task_id}')

    def ppo_agent(obs_dict):
        vec = obs_to_vector(obs_dict, max_steps)
        action, _ = model.predict(vec, deterministic=True)
        c, d = int_to_action(int(action))
        return {'concept': c, 'difficulty': d, 'hint_given': False, 'archetype_guess': None}

    print('=== PPO Agent Demo ===')
    env = TeachRLEnv(task_id=task_id, seed=99, eval_mode=True)
    obs = env.reset(seed=99)
    print(f'True archetype: {env._sim.archetype_id.value}')
    print(f'Expert hint: {obs.expert_hint}')
    print()

    done = False
    while not done:
        action = ppo_agent(obs.model_dump())
        result = env.step(action)
        obs    = result.observation
        done   = result.done

    print(env.render())
    print(f'\nFinal Score: {env._task_score():.4f}')

except FileNotFoundError:
    print('No trained model found. Run cells 8 first.')

## 13. Test the Live API

The environment is deployed as a live API on HuggingFace Spaces.

In [ ]:
import requests, json

BASE_URL = 'https://archedequation-teachrl.hf.space'

# Check health
r = requests.get(f'{BASE_URL}/')
print('Health:', r.json()['status'])
print('Tasks:', r.json()['tasks'])

# Start episode
r = requests.post(f'{BASE_URL}/reset',
                  json={'task_id': 'blind_teaching', 'seed': 42})
data = r.json()
sid  = data['session_id']
print(f'\nSession: {sid[:8]}...')
print(f'Task: {data["task"]["id"]} ({data["task"]["difficulty"]})')

# Take one step
r = requests.post(f'{BASE_URL}/step',
                  json={'session_id': sid, 'concept': 'algebra_basics',
                        'difficulty': 'medium', 'archetype_guess': 'anxious_perfectionist'})
step = r.json()
print(f'\nReward: {step["reward"]:.3f}  Done: {step["done"]}')
print(f'Task score: {step["info"]["task_score"]:.4f}')

## 14. Raw Training Metrics From This Run

In [ ]:
import json, numpy as np, os
import matplotlib.pyplot as plt
os.chdir('/content/TeachRL')

tasks  = ['archetype_identification','adaptive_curriculum','blind_teaching','self_play_escalation']
colors = ['#27ae60','#f39c12','#e74c3c','#8e44ad']
max_ep_rewards = {'archetype_identification': 10.0, 'adaptive_curriculum': 20.0,
                  'blind_teaching': 40.0, 'self_play_escalation': 40.0}

def smooth(x, w=5):
    x = np.array(x, dtype=float)
    return np.convolve(x, np.ones(w)/w, mode='valid') if len(x)>w else x

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('TeachRL — Raw Training Metrics From This Colab Run\n'
             'x: training steps (thousands) | y: mean episode reward [0–1]',
             fontsize=13, fontweight='bold')

for ax, task_id, color in zip(axes.flat, tasks, colors):
    path = f'training_plots/data/training_{task_id}.json'
    if os.path.exists(path):
        with open(path) as f: d = json.load(f)
        steps = np.array(d['steps'], dtype=float)
        rew   = np.array(d['rewards'], dtype=float)

        # Normalize to [0,1] if values are cumulative episode rewards
        if rew.max() > 1.5:
            rew = rew / max_ep_rewards.get(task_id, rew.max())
        rew = np.clip(rew, 0, 1)

        sm = smooth(rew, w=min(5, len(rew)))
        ts = steps[:len(sm)]

        ax.plot(steps/1000, rew, color=color, alpha=0.25, linewidth=1, label='Raw')
        ax.fill_between(ts/1000, np.clip(sm-0.02,0,1), np.clip(sm+0.02,0,1),
                        alpha=0.15, color=color)
        ax.plot(ts/1000, sm, color=color, linewidth=2.5, label='Smoothed')
        ax.text(0.02, 0.97,
                f'Log points: {len(steps)}\nMin: {rew.min():.3f}\nMax: {rew.max():.3f}\nFinal: {rew[-1]:.3f}',
                transform=ax.transAxes, fontsize=9, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

        short = {'archetype_identification':'Easy','adaptive_curriculum':'Medium',
                 'blind_teaching':'Hard','self_play_escalation':'Expert'}[task_id]
        ax.set_title(f'{short}: {task_id.replace("_"," ").title()}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Training Steps (thousands)')
        ax.set_ylabel('Normalized Reward [0–1]')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    else:
        ax.text(0.5, 0.5, 'No data\nRun training first',
                ha='center', va='center', transform=ax.transAxes, color='#aaa')
        ax.set_title(task_id)

plt.tight_layout()
plt.savefig('training_plots/training_metrics_this_run.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved!')

## 15 . Score Table From This Run:

In [ ]:
import json, os
os.chdir('/content/TeachRL')

tasks  = ['archetype_identification','adaptive_curriculum','blind_teaching','self_play_escalation']
agents = ['Random','Heuristic','Greedy','Inference','PPO+Clf']
scores = {a: [] for a in agents}
shorts = {'archetype_identification':'Easy','adaptive_curriculum':'Medium',
          'blind_teaching':'Hard','self_play_escalation':'Expert'}

for task_id in tasks:
    path = f'training_plots/data/eval_{task_id}.json'
    if os.path.exists(path):
        with open(path) as f: d = json.load(f)
        for a in agents: scores[a].append(d.get(a,{}).get('score',0.0))
    else:
        for a in agents: scores[a].append(None)

print(f'\n{"Agent":<16}', end='')
for t in tasks: print(f'{shorts[t]:>12}', end='')
print('\n' + '─'*64)
for agent in agents:
    print(f'{agent:<16}', end='')
    for s in scores[agent]:
        print(f'{s:>12.3f}' if s is not None else f'{"—":>12}', end='')
    print(' ← 🏆' if agent == 'PPO+Clf' else '')

ppo   = scores['PPO+Clf']
bests = [max(scores[a][i] for a in agents if a!='PPO+Clf' and scores[a][i]) for i in range(4)]
deltas = [p-b if p else None for p,b in zip(ppo,bests)]
print('\nPPO vs best baseline:')
for t, d in zip(tasks, deltas):
    if d is not None:
        mark = '✅' if d > 0 else '❌'
        print(f'  {mark} {shorts[t]}: {"+".join([""])+f"{d:.4f}"}')
print(f'\nPPO wins on {sum(1 for d in deltas if d and d>0)}/4 tasks')

## 16 .Self-Improvement Plot From This Run:

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
os.chdir('/content/TeachRL')

ARCH_COLORS = {
    'overconfident_learner':'#e74c3c','anxious_perfectionist':'#9b59b6',
    'adhd_sprinter':'#3498db','slow_steady_builder':'#27ae60',
    'strategic_gamer':'#f39c12','emotional_learner':'#1abc9c',
    'uneven_genius':'#e67e22','impostor':'#34495e',
}

path = 'training_plots/data/self_play_data.json'
if not os.path.exists(path):
    print('Run Step 8 first (collect_self_play_data)')
else:
    with open(path) as f: sp = json.load(f)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        f'Self-Improvement Evidence — Agent: {sp["agent"]}\n'
        f'Episodes: {sp["episodes"]}  |  Total Escalations: {sp["total_escalations"]}',
        fontsize=13, fontweight='bold')

    eps = list(range(1, sp['episodes']+1))
    ax1.plot(eps, sp['scores'], '-', color='#2c3e50', linewidth=2, zorder=3)
    for ep, sc, arch in zip(eps, sp['scores'], sp['archetypes']):
        ax1.scatter(ep, sc, color=ARCH_COLORS.get(arch,'#999'),
                    s=130, zorder=5, edgecolors='white', linewidth=1.5)
    for ev in sp['escalation_events']:
        ax1.axvline(ev['episode'], color='#e74c3c', linestyle='--', linewidth=2, alpha=0.8)
        ax1.annotate(
            f'ESCALATION\n{ev["archetype"][:12]}\nGen→{ev["new_gen"]}',
            xy=(ev['episode'], ev['score']),
            xytext=(ev['episode']+0.2, min(ev['score']+0.12, 0.96)),
            fontsize=8, color='#c0392b', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='#e74c3c', alpha=0.9),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.2))
    ax1.axhline(0.70, color='#e74c3c', linestyle=':', linewidth=1.5, alpha=0.5)
    ax1.text(eps[-1]-0.5, 0.71, 'escalation threshold', fontsize=8, color='#e74c3c')
    ax1.set_xlabel('Episode Number', fontsize=11)
    ax1.set_ylabel('Task Score [0–1]', fontsize=11)
    ax1.set_title('Score Per Episode — coloured by archetype\nRed dashed = escalation triggered',
                  fontsize=11, fontweight='bold')
    ax1.set_ylim(0, 1.1); ax1.grid(True, alpha=0.2)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    gens  = sp['final_generations']
    names = [k.replace('_','\n') for k in gens]
    vals  = list(gens.values())
    bars  = ax2.bar(names, vals,
                    color=[ARCH_COLORS.get(k,'#999') for k in gens],
                    alpha=0.85, edgecolor='white')
    for bar, g in zip(bars, vals):
        if g > 0:
            ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
                     f'Gen {g}', ha='center', fontsize=9,
                     fontweight='bold', color='#c0392b')
    ax2.set_ylabel('Escalation Generation', fontsize=11)
    ax2.set_title('Escalations Per Archetype\nGen > 0 = env got harder',
                  fontsize=11, fontweight='bold')
    ax2.set_ylim(0, max(vals)+1.5 if max(vals)>0 else 2)
    ax2.tick_params(axis='x', labelsize=7)
    ax2.grid(True, axis='y', alpha=0.2)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    n = len(sp['scores']); half = n//2
    early = sum(sp['scores'][:half])/half
    late  = sum(sp['scores'][half:])/half
    print(f'Early avg: {early:.4f}  →  Late avg: {late:.4f}  (delta: {late-early:+.4f})')
    print(f'Escalations: {sp["escalation_events"]}')

    plt.tight_layout()
    plt.savefig('training_plots/escalation_this_run.png', dpi=150, bbox_inches='tight')
    plt.show()

## 17. Testinf the live HF Space

In [ ]:
import requests
BASE = 'https://archedequation-teachrl.hf.space'

# Health check
r = requests.get(f'{BASE}/')
print('Status:', r.json()['status'])
print('Tasks:', r.json()['tasks'])
print('Archetypes:', r.json()['archetypes'])

# Start episode
r   = requests.post(f'{BASE}/reset', json={'task_id':'blind_teaching','seed':42})
sid = r.json()['session_id']
print(f'\nSession: {sid[:8]}...')
print(f'Task: {r.json()["task"]["id"]} ({r.json()["task"]["difficulty"]})')
print(f'Max steps: {r.json()["task"]["max_steps"]}')

# Take 5 steps
actions = [
    {'concept':'algebra_basics',   'difficulty':'medium', 'archetype_guess':'overconfident_learner'},
    {'concept':'linear_equations', 'difficulty':'easy',   'archetype_guess':None},
    {'concept':'functions',        'difficulty':'hard',   'archetype_guess':None},
    {'concept':'probability',      'difficulty':'medium', 'archetype_guess':None},
    {'concept':'geometry',         'difficulty':'easy',   'archetype_guess':None},
]

print(f'\n{"Step":<6}{"Concept":<24}{"Diff":<8}{"Reward":>8}{"Score":>8}{"Done":>7}')
print('─'*63)
for i, act in enumerate(actions):
    payload = {'session_id':sid, 'hint_given':False, **act}
    d = requests.post(f'{BASE}/step', json=payload).json()
    print(f'{i+1:<6}{act["concept"]:<24}{act["difficulty"]:<8}'
          f'{d["reward"]:>8.3f}{d["info"]["task_score"]:>8.3f}{str(d["done"]):>7}')

# State snapshot
state = requests.get(f'{BASE}/state', params={'session_id':sid}).json()
print(f'\nSteps remaining: {state["steps_remaining"]}')
print(f'Cumulative reward: {state["cumulative_reward"]:.3f}')

print('\nSwagger UI: https://archedequation-teachrl.hf.space/docs')
print('Try it yourself: POST /reset then POST /step')